# NB-01 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-01**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-01.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This computes the three-bus clearing example from Volume I, then checks reference-bus invariance, marginal prices, and the settlement ledger.

- [Volume I, §5.1 — The clearing LP and its duals](https://sherandoresearch.github.io/power-markets-series/vol1_congestion_lmp.html#s5-1)
- [Volume I, §4.3 — Reference-bus dependence and gauge invariance](https://sherandoresearch.github.io/power-markets-series/vol1_congestion_lmp.html#s4-3)
- [Volume I, §5.4 — Congestion rent and settlement](https://sherandoresearch.github.io/power-markets-series/vol1_congestion_lmp.html#s5-4)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-01', require_key=True)
print(setup_status)


# NB-01 — One triangle: flows, dispatch, and prices
Companion to Volume I §§4–6. Requires basic matrix multiplication and LP
duality. Reproduce the existing equal-reactance model and the published
300 MW base case; no new synthetic scenario is introduced.

Line order is 1→2, 1→3, 2→3. Positive injection means generation minus load.
Prices are USD/MWh, flows MW, and cost USD for a one-hour interval.

In [ ]:
import copy
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot, read_saved
session = Session('NB-01', version='1.43.0')
print('Private input contract:', session.input_manifest)
network_request = session.inputs['network']
dispatch_request = session.inputs['dispatch']
print('Network inputs:', network_request['inputs'])
print('Dispatch inputs:', dispatch_request['inputs'])
def network(injection):
    request = copy.deepcopy(network_request)
    request['inputs']['balanced_injections']['values'] = np.asarray(injection).tolist()
    return session.execute('dc_network_response', request)
def flow(injection):
    return np.asarray(network(injection)['result']['flow_field']['values'][0])
def dispatch(demand=None):
    request = copy.deepcopy(dispatch_request)
    if demand is not None:
        request['inputs']['demand_and_limits']['demand_mw'] = np.asarray(demand).tolist()
    r = session.execute('dc_dispatch', request)['result']['dispatch_result']
    return {'generation_mw':np.asarray(r['generation_mw']), 'flow_mw':np.asarray(r['flows_mw']),
        'lmp_usd_mwh':np.asarray(r['nodal_prices']),
        'directional_duals':np.asarray(r['upper_duals'] + r['lower_duals']),
        'signed_dual':r['upper_duals'][0]-r['lower_duals'][0], 'cost_usd_hour':r['interval_cost'],
        'demand_mw':np.asarray(request['inputs']['demand_and_limits']['demand_mw'])}
transaction = np.array([1., 0., -1.])
network_response = network(transaction)
A = np.asarray(network_request['inputs']['electrical_network']['incidence'])
psi = np.asarray(network_response['result']['network_operator_bundle']['shift_factor_operator'])
base_flow = np.asarray(network_response['result']['flow_field']['values'][0])
print('Slack-free PTDF, rows [1→2, 1→3, 2→3]:\n', psi.round(6))
print('Published 1 MW transfer:', base_flow)
np.testing.assert_allclose(base_flow, [1/3, 2/3, 1/3])
np.testing.assert_allclose(A.T @ base_flow, transaction, atol=1e-10)

## Dispatch the published congested base case
Bus 3 consumes 300 MW. Generators at buses 1 and 2 offer at $10 and $30/MWh.
Only line 1–3 is limited, to 150 MW. The expensive generator creates
counterflow relative to bus 1's supply, allowing the load to be served.

In [ ]:
result = dispatch()
for key, value in result.items():
    print(key, np.round(value, 6))
np.testing.assert_allclose(result['generation_mw'], [150,150], atol=1e-8)
np.testing.assert_allclose(result['lmp_usd_mwh'], [10,30,50], atol=1e-8)
assert abs(result['signed_dual']-60)<1e-8
# The same three grounded solves now run inside the governed private method.
grounded = network_response['diagnostics']['grounded_reference_checks']
assert [r['reference_node_index'] for r in grounded] == [0,1,2]
for row in grounded:
    assert row['angles'][row['reference_node_index']] == 0
    np.testing.assert_allclose(row['flows'], base_flow, atol=1e-10)
print('Grounded-reference certificates:', grounded)

## Check marginal prices and the settlement ledger
A small finite-difference derivative at this same operating point checks
the LP's dual prices. This numerical check is not an additional data series.
The load pays $15,000; generators receive $6,000; congestion rent is $9,000.

In [ ]:
epsilon = 1e-4
marginals = []
for bus in range(3):
    demand = result['demand_mw'].copy()
    demand[bus] += epsilon
    marginals.append((dispatch(demand)['cost_usd_hour']-result['cost_usd_hour'])/epsilon)
np.testing.assert_allclose(marginals, result['lmp_usd_mwh'], atol=1e-5)
load_payment = result['demand_mw'] @ result['lmp_usd_mwh']
generator_payment = result['generation_mw'] @ result['lmp_usd_mwh'][:2]
rent = load_payment-generator_payment
assert abs(rent-150*result['signed_dual'])<1e-7
print({'load_payment_usd': load_payment, 'generator_payment_usd': generator_payment, 'congestion_rent_usd': rent})
plt = setup_plot()
fig, ax = plt.subplots()
ax.bar(['Bus 1','Bus 2','Bus 3'], result['lmp_usd_mwh'], color=['#88C0D0','#81A1C1','#B48EAD'])
ax.set(ylabel='USD/MWh', title='Published triangle: one constraint separates three prices')
fig.tight_layout()
plt.show()

## Exercise and failure case
Reverse the 1→3 transaction. All line flows must reverse sign. Then try
an unbalanced injection: a slack-free PTDF must not hide an unstated source.

In [ ]:
np.testing.assert_allclose(flow(-transaction), -base_flow)
try:
    flow([1,0,0])
except MethodsError as exc:
    assert 'HTTP 422' in str(exc)
    print('Expected remote rejection:', exc)
else:
    raise AssertionError('Unbalanced injection accepted')

In [ ]:
summary = session.finish(
    ['published dispatch/prices', 'KCL', 'grounded gauge checks', 'marginal prices', 'settlement identity', 'imbalance rejected'],
    {'scenario':'Volume I 5.1 existing base case'})
print(summary)
# Explicit local access remains available without executing a method again.
saved = read_saved(session.output / session.calls[0]['result_file'])
assert saved['receipt']['result_sha256'] == network_response['receipt']['result_sha256']

## What this does not establish
This is the book's lossless continuous DC model. It does not model commitment,
losses, funding shortfalls, or an actual ISO grid. Nonnegative directional
inequality duals and a signed net constraint coefficient are distinct objects.